# 多模态核心能力开发与代码实现

---

## 课程导航

| 部分 | 内容 | 对应文件 |
|------|------|----------|
| **Part 1** | 本节在架构中的位置 · 感知层 | — |
| **Part 2** | 环境准备与模块结构 | `core/rag/multimodal.py` |
| **Part 3** | ASR：从"写死平台分支"到引擎抽象 | `core/rag/multimodal.py` |
| **Part 4** | TTS：把"会说话"封装成一条链路 | `core/rag/multimodal.py` |
| **Part 5** | OCR + VLM：读图与看懂图 | `core/rag/multimodal.py` |
| **Part 6** | 媒体全生命周期：降采样 → 落盘 → 鉴权访问 | `core/infra/media.py`、`core/rag/parsers.py` |
| **Part 7** | 完整链路 + 随堂练习（含答案） | 多模块联合 |

---

> **面向学员**：本教程面向弱基础学员，每个知识点都有详细讲解和代码注释，
> 建议按顺序逐 Cell 运行，观察每一次的输出结果。
>
> 本节课不是"背 API"，而是回答一个问题：
> **把"能听 / 能说 / 能看 / 读图"做成一条可复用的工具层，代码为什么要这么设计？**

---
## Part 1：本节在架构中的位置 —— 感知层
---

### 1.1 先看一条完整的数据流

本项目是 **IT 服务台** 场景：员工用【文字 / 语音 / 截图】提问，AI 数字员工自助解决，
解决不了就转人工坐席或自动建工单。

```
┌────────────┐   ┌──────────────┐   ┌──────────┐   ┌────────────┐   ┌────────────┐
│ 感知(用户输入) │ → │ 意图 + 情绪识别 │ → │ 知识库检索 │ → │ 多 Agent 编排 │ → │ 工单 / 人机协同 │
│ 语音 / 截图   │   │ perceive_*   │   │   RAG    │   │  CrewAI    │   │ 前端实时推送  │
└────────────┘   └──────────────┘   └──────────┘   └────────────┘   └────────────┘
```

本节课专门讲**最左边那一格**：如何把"听不懂的语音、看不懂的截图"变成下游能消费的文本与结构化结果。

### 1.2 感知层在五层架构中的位置

| 层 | 目录 | 本节涉及的代码 |
|---|---|---|
| 路由层 | `core/api/` | `chat_api.py`：`/api/media/upload`、`/api/media/{id}` |
| 基础设施层 | `core/infra/` | `media.py`：媒体落盘、寻址、鉴权 |
| 智能编排层 | `core/ai/` | `llm.py`：统一模型客户端、`extract_json()` |
| 多模态 RAG 层 | `core/rag/` | `multimodal.py`（ASR/OCR/VLM/TTS/情绪）、`parsers.downsample()` |
| 业务编排层 | `core/service/` | 消费感知结果（后续课时讲解） |

> 教学意义：感知层是"最贴近大模型"的一条工具链，它把平台差异、供应商差异全部关起来，
> 对上层只暴露几个简单函数。这样上层业务才能不关心"用户是打字还是说话"。

### 1.3 四条（加两条）感知链路

| 链路 | 输入 → 输出 | 对外函数 |
|------|------------|----------|
| ASR 语音转文字 | 语音 → 文本 | `process_voice()` |
| OCR 图内文字 | 图片 → 文字 | `_ocr_local_sync()`（内部） |
| VLM 截图诊断 | 截图 → 结构化诊断 | `process_image()` |
| VLM 配图描述 | 知识库配图 → 一段描述 | `describe_knowledge_image()` |
| 会话前置识别 | 文本 → 意图 + 情绪 | `perceive_message()` |
| TTS 文字转语音 | 文本 → 音频 | `synthesize_speech()` |

> 本项目聚焦**理解类多模态**（ASR / OCR / VLM）与**表达类**（TTS），
> 不做"文生图"——IT 服务台没有生成配图的需求。

### 1.4 三条贯穿全节的统一约定

1. **对外只暴露 async 函数**：底层 SDK 是同步的，用 `asyncio.to_thread()` 包一层，避免阻塞事件循环。
2. **失败抛 `PerceptionError`（中文可读），绝不返回错误文案**：否则调用方会把"语音识别失败…"当成用户说的话落库、送检索。
3. **统一「能力四元组」配置**：换供应商只改 `.env`，代码零改动。

---
## Part 2：环境准备与模块结构
---

### 2.1 检查 Python 环境与依赖

In [1]:
# ============================================================
# Cell 1：环境与依赖检查
# ============================================================
import sys
import subprocess
import importlib

print("=" * 55)
print("  Python 环境检查")
print("=" * 55)
print(f"  Python 版本: {sys.version}")
print(f"  运行平台:   {sys.platform}")
print(f"  解释器路径: {sys.executable}")
print()

# 把项目根目录加入 import 路径，保证能 import core.*
sys.path.insert(0, '.')
print(f"  当前工作目录: {sys.path[0]}")
print()

required_packages = {
    'openai': 'openai',        # 统一模型客户端（ASR/VLM/LLM 都走它）
    'edge-tts': 'edge_tts',    # 本地免费 TTS
    'Pillow': 'PIL',           # 图片降采样
}

print("  依赖检查：")
for pkg_name, import_name in required_packages.items():
    try:
        importlib.import_module(import_name)
        print(f"     [OK] {pkg_name} 已安装")
    except ImportError:
        print(f"     [NO] {pkg_name} 未安装，正在安装...")
        try:
            r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg_name],
                               capture_output=True, text=True, timeout=180)
            print(f"     {'[OK] 安装成功' if r.returncode == 0 else '[NO] 安装失败: ' + r.stderr[:150]}")
        except Exception as e:
            print(f"     [NO] 安装失败: {e}")
print()
print("  环境检查完成。")

# ------------------------------------------------------------------
# 兼容补丁：让 asyncio.run() 在"已运行事件循环"的内核里也能用（幂等·可重入）
# ------------------------------------------------------------------
# 背景：部分 Jupyter / IDE 内核会把单元格代码放在一个**正在运行的事件循环**里执行，
# 此时直接调用 asyncio.run() 会报：
#     RuntimeError: asyncio.run() cannot be called from a running event loop
# 解决：包一层 run_async()——
#   · 当前线程【没有】运行中的事件循环：等价 asyncio.run（常规 Jupyter 行为不变）；
#   · 当前线程【已有】运行中的事件循环：把协程丢到"独立线程 + 新事件循环"执行，
#     再同步取回结果（线程内没有运行中的循环，故 asyncio.run 可正常启动）。
#
# 【为什么要写成幂等】本单元可能被复制/重复执行（例如编辑器把旧单元又贴回来）。
# 若把"原始实现"存成脚本全局变量，第二次执行会把它重绑成"上一次的补丁函数"，
# 于是在线程里调用它 → 自己调自己 → RecursionError。
# 故这里：① 用 asyncio 模块上的私有属性保存原始实现（不会被重绑覆盖）；
#          ② 加补丁标记 _mde_run_async_patched，重复执行直接跳过。
import asyncio
import concurrent.futures


def _install_asyncio_run_compat() -> bool:
    """安装兼容补丁；已安装则返回 False（不重复打补丁）。"""
    if getattr(asyncio, "_mde_run_async_patched", False):
        return False

    asyncio._mde_orig_run = asyncio.run   # 原始实现挂在模块属性上，后续不会被覆盖

    def run_async(coro, **kwargs):
        """在 notebook 中安全执行协程（兼容"已运行事件循环"的内核）。"""
        orig = asyncio._mde_orig_run      # 每次都取模块上的原始实现，确保不递归
        try:
            asyncio.get_running_loop()
        except RuntimeError:
            return orig(coro, **kwargs)   # 常规路径：无运行中的循环，直接跑
        # 兼容路径：已有运行中的循环 -> 换到独立线程的新循环里跑
        with concurrent.futures.ThreadPoolExecutor(max_workers=1) as pool:
            return pool.submit(lambda: orig(coro, **kwargs)).result()

    asyncio.run = run_async
    asyncio._mde_run_async_patched = True
    return True


if _install_asyncio_run_compat():
    print("已启用 run_async 兼容：asyncio.run() 现在可用于运行中的事件循环内核。")
else:
    print("run_async 兼容已启用（跳过重复补丁）。")


  Python 环境检查
  Python 版本: 3.13.7 (tags/v3.13.7:bcee1c3, Aug 14 2025, 14:15:11) [MSC v.1944 64 bit (AMD64)]
  运行平台:   win32
  解释器路径: c:\Users\13691\Documents\Code\260601-GPKJ\07-MLM\.venv\Scripts\python.exe

  当前工作目录: .

  依赖检查：
     [OK] openai 已安装
     [OK] edge-tts 已安装
     [OK] Pillow 已安装

  环境检查完成。
已启用 run_async 兼容：asyncio.run() 现在可用于运行中的事件循环内核。


> 说明：`paddleocr` / `funasr` 是**可选依赖**（本地 ASR/OCR 才需要）。
> 未安装时把 `ASR_PROVIDER` / `OCR_PROVIDER` 设为 `api`，会走在线模型，不影响本节学习。

### 2.2 先看模块的"说明书"

项目里的每个模块都有 `docstring`，它就是这个模块的"对外说明书"。先读说明书，再读代码。

In [2]:
# ============================================================
# Cell 2：查看感知层模块的 docstring 与对外接口
# ============================================================
import core.rag.multimodal as mm

print("=" * 55)
print("  core/rag/multimodal.py —— 模块说明书")
print("=" * 55)
print(mm.__doc__)
print("=" * 55)
print("  对外暴露的接口（__all__）：")
for name in mm.__all__:
    print(f"     - {name}")

  core/rag/multimodal.py —— 模块说明书
多模态感知：ASR / OCR / 视觉理解 / 情绪识别 / TTS，见设计文档 8.1–8.4。

对外接口：
  - :func:`process_voice`  —— 语音转文字（ASR）
  - :func:`process_image`  —— 图片 OCR + 视觉理解（结构化视觉诊断，被 ScreenshotAnalyzeTool 调用）
  - :func:`describe_knowledge_image` —— 知识库配图描述（入库期，写入文本空间作主召回）
  - :func:`perceive_message` —— 会话前置识别（一次调用同时产出「来意 intent」与「情绪」）
  - :func:`synthesize_speech` —— 文字转语音（edge-tts）

所有网络调用均有超时（30s）与异常兜底，关键错误以中文日志打印。

  对外暴露的接口（__all__）：
     - PerceptionError
     - process_voice
     - process_image
     - describe_knowledge_image
     - perceive_message
     - synthesize_speech
     - INTENT_ENUM


In [3]:
# ============================================================
# Cell 3：模块里的函数清单（先有地图，再看细节）
# ============================================================
import inspect

funcs = [
    (name, obj)
    for name, obj in vars(mm).items()
    if inspect.isfunction(obj) and obj.__module__ == mm.__name__
]
funcs.sort(key=lambda x: x[1].__code__.co_firstlineno)

print("=" * 55)
print(f"  共 {len(funcs)} 个函数（按源码顺序）")
print("=" * 55)
for name, obj in funcs:
    is_async = "async " if inspect.iscoroutinefunction(obj) else "       "
    print(f"  {is_async}{name:<26} (第 {obj.__code__.co_firstlineno} 行)")
print()
print("  规律：以 _ 开头的是内部实现（同步），不带 _ 的是对外接口（async）。")

  共 12 个函数（按源码顺序）
         _asr_api_sync              (第 92 行)
         _asr_local_sync            (第 107 行)
  async process_voice              (第 136 行)
         _ocr_local_sync            (第 170 行)
         _vlm_sync                  (第 215 行)
         _describe_sync             (第 248 行)
  async describe_knowledge_image   (第 274 行)
  async process_image              (第 302 行)
         _perceive_sync             (第 379 行)
  async perceive_message           (第 401 行)
         _tts_sync                  (第 450 行)
  async synthesize_speech          (第 474 行)

  规律：以 _ 开头的是内部实现（同步），不带 _ 的是对外接口（async）。


---
## Part 3：ASR 语音转文字 —— 从"写死平台分支"到引擎抽象
---

### 3.1 原理铺垫：语音是怎么变成文字的？

一句话概括：

```
语音波形 → (特征提取) → 声学模型 → (解码搜索) → 语言模型 → 文本
```

- **传统混合模型**：声学模型 + 发音词典 + 语言模型，三件套拼起来，工程复杂；
- **端到端模型**（如 Paraformer / Whisper）：一个模型直接出文本，工程简单、效果更好。

> 作为业务开发者，你**不需要**自己训练模型，只需要知道一件事：
> 无论用哪个引擎，业务只关心"**稳定地拿到文本**"。

### 3.2 v0：最直觉的写法 —— 把平台判断写进业务

刚接到需求时，最容易写出的代码长这样：

In [4]:
# ============================================================
# Cell 4：v0 —— 业务里直接判断操作系统（反面教材，不真正执行）
# ============================================================

v0_code = """
def transcribe_v0(audio_path):
    import platform
    if platform.system() == "Windows":
        from funasr import AutoModel      # Windows 只能跑本地
        model = AutoModel(model="paraformer-zh")
        return model.generate(input=audio_path)[0]["text"]
    else:
        from openai import OpenAI         # 其它平台走在线
        client = OpenAI(base_url="...", api_key="...")
        resp = client.audio.transcriptions.create(model="...", file=open(audio_path, "rb"))
        return resp.text
"""
print(v0_code)
print("痛点清单：")
print("  1. 业务代码里出现 platform.system()，平台差异泄漏到业务层")
print("  2. 想改供应商，得改业务函数，违反开闭原则")
print("  3. 本地库没装就直接 ImportError，没有降级路径")
print("  4. 失败时一旦返回字符串提示，调用方会当成用户说的话落库")


def transcribe_v0(audio_path):
    import platform
    if platform.system() == "Windows":
        from funasr import AutoModel      # Windows 只能跑本地
        model = AutoModel(model="paraformer-zh")
        return model.generate(input=audio_path)[0]["text"]
    else:
        from openai import OpenAI         # 其它平台走在线
        client = OpenAI(base_url="...", api_key="...")
        resp = client.audio.transcriptions.create(model="...", file=open(audio_path, "rb"))
        return resp.text

痛点清单：
  1. 业务代码里出现 platform.system()，平台差异泄漏到业务层
  2. 想改供应商，得改业务函数，违反开闭原则
  3. 本地库没装就直接 ImportError，没有降级路径
  4. 失败时一旦返回字符串提示，调用方会当成用户说的话落库


### 3.3 演进思路：把差异"关进引擎"，业务只选 provider

正确的做法是：**业务只决定"用哪个 provider"，具体怎么调由引擎函数负责**。

```
业务层：provider = ASR_PROVIDER  →  选 api 还是 local
                     │
        ┌────────────┴────────────┐
        ▼                         ▼
  _asr_api_sync()           _asr_local_sync()
  （在线，开箱即用）          （本地，可离线，需装 funasr）
```

- 平台差异被关在两个 `_xxx_sync()` 内部，业务不再出现 `sys.platform` / `platform.system()`；
- 本地库缺失时抛"**需要安装 funasr，或改用 ASR_PROVIDER=api**"这样**可操作**的提示；
- 失败统一抛 `PerceptionError`，从源头杜绝"错误文案被当成内容"。

In [5]:
# ============================================================
# Cell 5：v1 —— 业务只选 provider（演进后的骨架）
# ============================================================

v1_code = """
async def process_voice(audio_bytes, filename="audio.webm"):
    provider = (settings.ASR_PROVIDER or "api").lower()
    if provider == "local":
        text = await asyncio.to_thread(_asr_local_sync, audio_bytes, filename)
    else:
        text = await asyncio.to_thread(_asr_api_sync, audio_bytes, filename)
    return text.strip()
"""
print(v1_code)
print("关键点：")
print("  - asyncio.to_thread(...)：同步 SDK 丢到线程池，不阻塞 async 事件循环")
print("  - 业务层看不到任何平台判断，只看 ASR_PROVIDER")


async def process_voice(audio_bytes, filename="audio.webm"):
    provider = (settings.ASR_PROVIDER or "api").lower()
    if provider == "local":
        text = await asyncio.to_thread(_asr_local_sync, audio_bytes, filename)
    else:
        text = await asyncio.to_thread(_asr_api_sync, audio_bytes, filename)
    return text.strip()

关键点：
  - asyncio.to_thread(...)：同步 SDK 丢到线程池，不阻塞 async 事件循环
  - 业务层看不到任何平台判断，只看 ASR_PROVIDER


### 3.4 走读项目最终代码

前面手写的 v1，就是项目里的真实设计。下面直接看源码。

In [6]:
# ============================================================
# Cell 6：在线 ASR 引擎 _asr_api_sync
# ============================================================
print("=" * 55)
print("  _asr_api_sync —— 在线引擎（OpenAI 兼容 /audio/transcriptions）")
print("=" * 55)
print(inspect.getsource(mm._asr_api_sync))
print("重点：")
print("  - 配置缺失时抛 PerceptionError，并在提示里说清'缺哪个变量、怎么补'")
print("  - 未单独配 ASR_API_KEY 时回退 SILICONFLOW_API_KEY（密钥复用策略）")
print("  - timeout=30s，避免网络卡死拖垮整个会话")

  _asr_api_sync —— 在线引擎（OpenAI 兼容 /audio/transcriptions）
def _asr_api_sync(audio_bytes: bytes, filename: str) -> str:
    if not (settings.ASR_BASE_URL and settings.ASR_MODEL and settings.asr_key):
        raise PerceptionError(
            "ASR 未配置：请设置 ASR_BASE_URL / ASR_MODEL / ASR_API_KEY"
            "（未单独配置 Key 时回退 SILICONFLOW_API_KEY）。"
        )
    client = OpenAI(base_url=settings.ASR_BASE_URL, api_key=settings.asr_key)
    resp = client.audio.transcriptions.create(
        model=settings.ASR_MODEL,
        file=(filename, audio_bytes),
        timeout=_REQUEST_TIMEOUT,
    )
    return (getattr(resp, "text", "") or "").strip()

重点：
  - 配置缺失时抛 PerceptionError，并在提示里说清'缺哪个变量、怎么补'
  - 未单独配 ASR_API_KEY 时回退 SILICONFLOW_API_KEY（密钥复用策略）
  - timeout=30s，避免网络卡死拖垮整个会话


In [7]:
# ============================================================
# Cell 7：本地 ASR 引擎 _asr_local_sync
# ============================================================
print("=" * 55)
print("  _asr_local_sync —— 本地引擎（funasr，可离线）")
print("=" * 55)
print(inspect.getsource(mm._asr_local_sync))
print("重点：")
print("  - 先 try import funasr：没装就抛'可操作'的提示，而不是崩在底层")
print("  - 写临时文件喂给模型，finally 里删除，避免临时文件堆积")

  _asr_local_sync —— 本地引擎（funasr，可离线）
def _asr_local_sync(audio_bytes: bytes, filename: str) -> str:
    try:
        import funasr  # noqa: F401
    except Exception as exc:
        raise PerceptionError(
            "本地 ASR 需要安装 funasr（pip install funasr），或改用 ASR_PROVIDER=api。"
        ) from exc

    from funasr import AutoModel  # type: ignore

    suffix = Path(filename).suffix or ".wav"
    with tempfile.NamedTemporaryFile(suffix=suffix, delete=False) as tmp:
        tmp.write(audio_bytes)
        temp_path = tmp.name
    try:
        model = AutoModel(model="paraformer-zh")
        result = model.generate(input=temp_path)
        if result and isinstance(result, list):
            return str(result[0].get("text", "")).strip()
        return ""
    except Exception as exc:
        raise PerceptionError(f"本地 ASR 执行失败：{exc}") from exc
    finally:
        try:
            os.remove(temp_path)
        except OSError:
            pass

重点：
  - 先 try import funasr：没装就抛'可操作'的提示，而不是崩在底层

In [8]:
# ============================================================
# Cell 8：对外接口 process_voice —— 边界处理全在这里
# ============================================================
print("=" * 55)
print("  process_voice —— 对外唯一入口")
print("=" * 55)
print(inspect.getsource(mm.process_voice))

  process_voice —— 对外唯一入口
async def process_voice(audio_bytes: bytes, filename: str = "audio.webm") -> str:
    """语音转文字（ASR）。

    识别失败一律抛 :class:`PerceptionError`（携带中文可读信息），**不返回错误文案**——
    否则调用方会把提示串误当成用户转写内容落库、送 LLM/检索。调用方（如
    ``session_flow.preflight``）捕获异常后提示用户重录；评测套件亦按失败计入。

    - ``ASR_PROVIDER=="api"``：走 OpenAI 兼容 /audio/transcriptions；
    - ``ASR_PROVIDER=="local"``：尝试 funasr。
    """
    if not audio_bytes:
        raise PerceptionError("音频内容为空，请重新录制。")

    provider = (settings.ASR_PROVIDER or "api").lower()
    try:
        if provider == "local":
            text = await asyncio.to_thread(_asr_local_sync, audio_bytes, filename)
        else:
            text = await asyncio.to_thread(_asr_api_sync, audio_bytes, filename)
    except PerceptionError:
        raise
    except Exception as exc:  # noqa: BLE001
        logger.error("语音识别调用失败：%s", exc)
        raise PerceptionError(f"语音识别调用失败：{exc}（请检查网络或 ASR 服务配置）") from exc

    text = (text or "").strip()
    if not text

### 3.5 三类边界，三种处理原则

| 边界 | 处理 | 为什么 |
|------|------|--------|
| 空音频 `b""` | 抛 `PerceptionError("音频内容为空，请重新录制。")` | 没有内容可识别，早点失败、提示明确 |
| 引擎调用失败 | 捕获后抛 `PerceptionError(中文原因 + 建议)` | 让调用方能提示用户"重录 / 检查网络" |
| 引擎返回空文本 | 抛 `PerceptionError("未返回有效文本…")` | 空文本落库无意义，还会污染 LLM 上下文 |

> ⚠️ **最反直觉的一条**：识别失败**不能返回错误提示字符串**。
> 项目里有一个专门的回归测试 `tests/test_multimodal_asr.py`，起因就是：
> `process_voice` 曾把"语音识别失败：…"当正常返回值，
> 调用方把它当成用户的转写内容落库、送检索，导致"用户说的话变成了错误提示"。

### 3.6 实跑一次真实语义（不需要联网）

下面用**项目里的真实函数**验证上面三类边界。为了不依赖网络，
我们临时把 `_asr_api_sync` 替换成一个假的实现（这就是 pytest 里 `monkeypatch` 干的事）。

In [9]:
# ============================================================
# Cell 9：实跑 ASR 边界语义（复现 tests/test_multimodal_asr.py）
# ============================================================
import asyncio

# 保存原值，跑完恢复，避免污染后续 Cell
_orig_settings = mm.settings
_orig_api_sync = mm._asr_api_sync


class _FakeSettings:
    ASR_PROVIDER = "api"


mm.settings = _FakeSettings()

print("=" * 55)
print("  边界一：空音频")
print("=" * 55)
try:
    asyncio.run(mm.process_voice(b""))
except mm.PerceptionError as e:
    print(f"  [OK] 抛出 PerceptionError：{e}")

print()
print("=" * 55)
print("  边界二：引擎返回空文本（只有空白）")
print("=" * 55)
mm._asr_api_sync = lambda audio, filename: "   "
try:
    asyncio.run(mm.process_voice(b"audio-bytes"))
except mm.PerceptionError as e:
    print(f"  [OK] 抛出 PerceptionError：{e}")

print()
print("=" * 55)
print("  边界三：引擎调用失败")
print("=" * 55)
def _boom(audio, filename):
    raise RuntimeError("network down")

mm._asr_api_sync = _boom
try:
    asyncio.run(mm.process_voice(b"audio-bytes"))
except mm.PerceptionError as e:
    print(f"  [OK] 抛出 PerceptionError：{e}")

print()
print("=" * 55)
print("  正常路径：返回值自动去空白")
print("=" * 55)
mm._asr_api_sync = lambda audio, filename: "  VPN 连不上 "
print(f"  输入: '  VPN 连不上 '  →  输出: {asyncio.run(mm.process_voice(b'audio-bytes'))!r}")

# 恢复模块状态
mm.settings = _orig_settings
mm._asr_api_sync = _orig_api_sync
print()
print("  已恢复模块原始状态。")

语音识别调用失败：network down


  边界一：空音频
  [OK] 抛出 PerceptionError：音频内容为空，请重新录制。

  边界二：引擎返回空文本（只有空白）
  [OK] 抛出 PerceptionError：语音识别未返回有效文本，请重新录制或改用文字输入。

  边界三：引擎调用失败
  [OK] 抛出 PerceptionError：语音识别调用失败：network down（请检查网络或 ASR 服务配置）

  正常路径：返回值自动去空白
  输入: '  VPN 连不上 '  →  输出: 'VPN 连不上'

  已恢复模块原始状态。


> 💡 **小结**：ASR 这一条链路的全部设计价值，就是让业务层写下
> `text = await process_voice(audio_bytes)` 这一行——不关心平台、不关心供应商、
> 不用处理"失败文案混入内容"。这就是"工具层"存在的意义。

---
## Part 4：TTS 文字转语音 —— 把"会说话"封装成一条链路
---

### 4.1 TTS 在服务台里的作用

- 把 AI 的回复**播报**出来（无障碍 / 免看屏幕场景）；
- 语音回复也是"多模态数字员工"表达能力的体现。

本项目用 `edge-tts`：**本地、免费、无需 API Key**，适合教学与私有化。

> 📌 **一个诚实的说明**：课程大纲里还提到"逐词时间戳（字幕对齐）"与"引擎注册表 + provider_chain 降级链"。
> 这在本项目当前代码中是**设计预留**：配置项 `TTS_PROVIDER` 已留好，
> 但 `multimodal.py` 目前只实现了单引擎（edge-tts）封装。
> 本节按**真实代码**讲解，不做"代码里有、实际没有"的演绎。

### 4.2 v0：把 edge-tts 直接写在业务里

最容易写的版本通常是"临时文件 + 不清理 + 同步阻塞"：

In [10]:
# ============================================================
# Cell 10：v0 —— 直接写死（反面教材）
# ============================================================
v0_code = """
def speak_v0(text):
    import edge_tts
    import asyncio
    communicate = edge_tts.Communicate(text, "zh-CN-XiaoxiaoNeural")
    await communicate.save("out.mp3")      # 写死在当前目录
    return "out.mp3"
"""
print(v0_code)
print("痛点：")
print("  1. 音色/文件路径写死，换音色要改业务")
print("  2. 临时文件留在项目根目录，越跑越多")
print("  3. 同步阻塞事件循环")
print("  4. 文本为空时仍会去请求，浪费一次网络往返")


def speak_v0(text):
    import edge_tts
    import asyncio
    communicate = edge_tts.Communicate(text, "zh-CN-XiaoxiaoNeural")
    await communicate.save("out.mp3")      # 写死在当前目录
    return "out.mp3"

痛点：
  1. 音色/文件路径写死，换音色要改业务
  2. 临时文件留在项目根目录，越跑越多
  3. 同步阻塞事件循环
  4. 文本为空时仍会去请求，浪费一次网络往返


### 4.3 走读项目最终代码

项目把 TTS 拆成两层：
- `_tts_sync()`：**引擎层**，只管"文本 → mp3 字节"，用临时文件并在 `finally` 里清理；
- `synthesize_speech()`：**接口层**，做参数校验（空文本早失败）、选音色、把同步实现丢线程池。

In [11]:
# ============================================================
# Cell 11：TTS 引擎层 _tts_sync
# ============================================================
print("=" * 55)
print("  _tts_sync —— 引擎层（文本 → mp3 字节）")
print("=" * 55)
print(inspect.getsource(mm._tts_sync))
print("重点：")
print("  - edge-tts 未安装时抛'可操作'提示（pip install edge-tts）")
print("  - 用 NamedTemporaryFile 落临时文件，finally 里 os.remove")
print("  - asyncio.run 在独立线程里驱动 edge-tts 的 async 接口")

  _tts_sync —— 引擎层（文本 → mp3 字节）
def _tts_sync(text: str, voice: str) -> bytes:
    try:
        import edge_tts
    except Exception as exc:
        raise PerceptionError(
            "TTS 依赖 edge-tts 未安装（pip install edge-tts）。"
        ) from exc

    async def _run() -> bytes:
        communicate = edge_tts.Communicate(text, voice)
        with tempfile.NamedTemporaryFile(suffix=".mp3", delete=False) as tmp:
            temp_path = tmp.name
        try:
            await communicate.save(temp_path)
            return Path(temp_path).read_bytes()
        finally:
            try:
                os.remove(temp_path)
            except OSError:
                pass

    return asyncio.run(_run())

重点：
  - edge-tts 未安装时抛'可操作'提示（pip install edge-tts）
  - 用 NamedTemporaryFile 落临时文件，finally 里 os.remove
  - asyncio.run 在独立线程里驱动 edge-tts 的 async 接口


In [12]:
# ============================================================
# Cell 12：TTS 接口层 synthesize_speech
# ============================================================
print("=" * 55)
print("  synthesize_speech —— 对外接口")
print("=" * 55)
print(inspect.getsource(mm.synthesize_speech))
print("重点：")
print("  - 空文本前置校验：'语音合成失败：待合成文本为空。'")
print("  - 音色可传入，缺省用 settings.TTS_VOICE")
print("  - 返回 (bytes, 'mp3')：扩展名由引擎返回，调用方不写死后缀")
print("    → 将来换引擎（比如输出 wav），调用方代码不用改")

  synthesize_speech —— 对外接口
async def synthesize_speech(text: str, voice: Optional[str] = None) -> tuple[bytes, str]:
    """文字转语音，返回 ``(mp3_bytes, "mp3")``。失败抛 :class:`PerceptionError`。"""
    if not text or not text.strip():
        raise PerceptionError("语音合成失败：待合成文本为空。")
    chosen_voice = voice or settings.TTS_VOICE
    try:
        audio = await asyncio.to_thread(_tts_sync, text, chosen_voice)
    except PerceptionError:
        raise
    except Exception as exc:
        logger.error("语音合成失败：%s", exc)
        raise PerceptionError(f"语音合成失败：{exc}") from exc
    return audio, "mp3"

重点：
  - 空文本前置校验：'语音合成失败：待合成文本为空。'
  - 音色可传入，缺省用 settings.TTS_VOICE
  - 返回 (bytes, 'mp3')：扩展名由引擎返回，调用方不写死后缀
    → 将来换引擎（比如输出 wav），调用方代码不用改


In [13]:
# ============================================================
# Cell 13：看看 TTS 相关配置与依赖现状
# ============================================================
from core.config import settings

print("=" * 55)
print("  TTS 配置")
print("=" * 55)
print(f"  TTS_PROVIDER: {settings.TTS_PROVIDER}   （当前仅 edge 一种实现）")
print(f"  TTS_VOICE:    {settings.TTS_VOICE}")
print(f"  edge-tts 已安装: {importlib.util.find_spec('edge_tts') is not None}")
print()
print("  提示：TTS_PROVIDER 是'为将来留的口子'——")
print("        想接第二家引擎时，把分支加在 synthesize_speech 内部即可，接口签名不变。")

  TTS 配置
  TTS_PROVIDER: edge   （当前仅 edge 一种实现）
  TTS_VOICE:    zh-CN-XiaoxiaoNeural
  edge-tts 已安装: True

  提示：TTS_PROVIDER 是'为将来留的口子'——
        想接第二家引擎时，把分支加在 synthesize_speech 内部即可，接口签名不变。


In [14]:
# ============================================================
# Cell 14（可选）：真实合成一段语音（需要 edge-tts + 网络）
# ============================================================
import asyncio

if importlib.util.find_spec('edge_tts') is None:
    print("  edge-tts 未安装，跳过真实合成。")
else:
    try:
        audio_bytes, ext = asyncio.run(mm.synthesize_speech("VPN 连接失败，请检查网络配置。"))
        print(f"  [OK] 合成成功：{len(audio_bytes)} 字节，扩展名 = {ext}")
        print(f"  前 4 个字节（MP3 头通常含 ID3/帧同步）：{audio_bytes[:4]!r}")
    except Exception as e:
        print(f"  [跳过] 合成失败（可能是网络不可达）：{e}")
        print("  这不影响本节学习——失败同样被封装成 PerceptionError。")

  [OK] 合成成功：23184 字节，扩展名 = mp3
  前 4 个字节（MP3 头通常含 ID3/帧同步）：b'\xff\xf3d\xc4'


### 4.4 TTS 的三个设计点

| 设计点 | 代码位置 | 好处 |
|--------|----------|------|
| 空文本早失败 | `synthesize_speech` 开头 | 省一次无意义的网络往返 |
| 扩展名由引擎返回 | 返回 `(bytes, "mp3")` | 换引擎不改调用方 |
| 临时文件必清理 | `_tts_sync` 的 `finally` | 不留"脏文件"，这是产品级细节 |

---
## Part 5：OCR + VLM —— 读图与看懂图
---

### 5.1 OCR 的定位：只负责"把图里的字抠出来"

要特别注意：**OCR 不是文档解析**。
- 文档解析（第 36 节）负责把 PDF/Word 变成结构化的页与块；
- OCR 只负责截图里"有哪些字"，是**截图理解的一个辅助步骤**。

项目提供两种 OCR 模式（`OCR_PROVIDER`）：
- `local`：本地 PaddleOCR（离线、免费，但要装依赖）；
- `api`：直接复用 VLM 一起完成（少装依赖，但走网络）。

### 5.2 走读 OCR 源码

In [15]:
# ============================================================
# Cell 15：本地 OCR 引擎 _ocr_local_sync
# ============================================================
print("=" * 55)
print("  _ocr_local_sync —— 本地 PaddleOCR")
print("=" * 55)
print(inspect.getsource(mm._ocr_local_sync))
print("重点：")
print("  - PaddleOCR 2.x / 3.x 接口不同，代码里做了两种调用方式的兼容")
print("  - 返回用换行拼起来的纯文本，交给 VLM 做'校正/理解'")

  _ocr_local_sync —— 本地 PaddleOCR
def _ocr_local_sync(image_bytes: bytes) -> str:
    try:
        from paddleocr import PaddleOCR
    except Exception as exc:
        raise PerceptionError(
            "本地 OCR 需要安装 PaddleOCR（pip install paddleocr paddlepaddle），"
            "或设置 OCR_PROVIDER=api 复用 VLM。"
        ) from exc

    try:
        import numpy as np
        from PIL import Image

        image = Image.open(io.BytesIO(image_bytes)).convert("RGB")
        array = np.array(image)
        ocr = PaddleOCR(use_angle_cls=True, lang="ch", show_log=False)

        lines: list[str] = []
        try:
            result = ocr.ocr(array, cls=True)
        except (AttributeError, TypeError):
            # PaddleOCR 3.x 的 predict 接口
            result = ocr.predict(array)
            for item in result or []:
                if isinstance(item, dict):
                    lines.extend(str(t) for t in item.get("rec_texts", []) or [])

        if not lines:
            for block in result or 

### 5.3 VLM 的两类用途：同一个模型，两种提示词

这是本节最重要的设计对比。VLM（视觉语言模型）在本项目里有**两种完全不同的用途**：

| 用途 | 调用时机 | 提示词 | 输出风格 | 失败时 |
|------|---------|--------|---------|--------|
| **截图诊断** | 用户会话中 | `_VLM_PROMPT` | 面向诊断，输出 `error_code / error_type` 等结构化 JSON | 返回带错误说明的 dict |
| **知识库配图描述** | 知识入库时 | `_KB_IMAGE_PROMPT` | 面向检索，输出一段**客观中立**的中文描述 | 返回空串，绝不阻断入库 |

> 为什么入库期（配图描述）失败返回空串、而会话期（截图诊断）失败返回错误 dict？
> 因为两者对"失败"的容忍度不同：
> - **入库是大批量批处理**，一张图描述不出来，回退页正文继续入库，不能因一张图中断整批；
> - **会话是即时交互**，诊断失败必须告诉用户"这次没看懂，可以重试/改文字"，不能静默。

下面直接打印两个真实提示词的开头，感受差异。

In [16]:
# ============================================================
# Cell 16：对比两个 VLM 提示词（真实字符串）
# ============================================================
print("=" * 55)
print("  提示词一：_VLM_PROMPT（截图诊断，面向结构化输出）")
print("=" * 55)
print(mm._VLM_PROMPT[:260], "...")
print()
print("=" * 55)
print("  提示词二：_KB_IMAGE_PROMPT（配图描述，面向检索）")
print("=" * 55)
print(mm._KB_IMAGE_PROMPT[:260], "...")
print()
print("  观察差异：")
print("    - 前者要求'只输出一个 JSON 对象'，字段固定；")
print("    - 后者要求'不超过 120 字的一段中文''不要输出 JSON'，且禁止臆测。")

  提示词一：_VLM_PROMPT（截图诊断，面向结构化输出）
你是 IT 服务台的截图诊断助手。请分析用户上传的报错截图 / 界面截图。
本地 OCR 已识别的文字（可能为空或不准确，请结合图片自行校正）：
{ocr}

请只输出一个 JSON 对象，不要输出任何多余文字，字段如下：
{{
  "ocr_text": "图中关键文字（错误码、报错信息、配置项等）",
  "description": "对截图内容与场景的一句话中文描述",
  "error_code": "错误码（如 502、619 等，无则空字符串）",
  "service": "涉及的服务/组件（如 n ...

  提示词二：_KB_IMAGE_PROMPT（配图描述，面向检索）
你是企业IT知识库的配图标注助手。请客观描述这张知识库插图，供后续检索与作答使用。
只输出一段中文（不超过120字），依次包含：
1) 图类型（网络拓扑 / 流程图 / 界面截图 / 报错截图 / 照片 / 图表等）；
2) 主体与关键组件或系统（如 VPN 网关、域控、OA、打印机）；
3) 图中可见的关键文字、标签或参数（如 IP、域名、错误码、按钮）；
4) 该图表达的关键关系或结论（如访问路径、故障点、配置位置）。
不要臆测图中不存在的信息；不确定的项可省略。不要输出 JSON。 ...

  观察差异：
    - 前者要求'只输出一个 JSON 对象'，字段固定；
    - 后者要求'不超过 120 字的一段中文''不要输出 JSON'，且禁止臆测。


In [17]:
# ============================================================
# Cell 17：VLM 引擎 _vlm_sync（截图诊断）
# ============================================================
print("=" * 55)
print("  _vlm_sync —— 视觉理解（截图诊断）")
print("=" * 55)
print(inspect.getsource(mm._vlm_sync))
print("重点：")
print("  - 图片以 data_url 形式放进 messages 的 image_url 字段（多模态消息格式）")
print("  - 用 extract_json() 容错解析；解析失败则把整段文本当 description 兜底")
print("  - 本地 OCR 的结果作为文字一起喂给 VLM，让模型'校正 + 理解'")

  _vlm_sync —— 视觉理解（截图诊断）
def _vlm_sync(data_url: str, ocr_text: str) -> dict:
    if not (settings.VLM_BASE_URL and settings.VLM_MODEL and settings.vlm_key):
        raise PerceptionError(
            "视觉理解未配置：请设置 VLM_BASE_URL / VLM_MODEL / VLM_API_KEY"
            "（未单独配置 Key 时回退 SILICONFLOW_API_KEY / LLM_API_KEY）。"
        )
    client = OpenAI(base_url=settings.VLM_BASE_URL, api_key=settings.vlm_key)
    prompt = _VLM_PROMPT.format(
        ocr=ocr_text or "（本地 OCR 未提供文本，请自行识别图中文字）"
    )
    resp = client.chat.completions.create(
        model=settings.VLM_MODEL,
        messages=[
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": prompt},
                    {"type": "image_url", "image_url": {"url": data_url}},
                ],
            }
        ],
        timeout=_REQUEST_TIMEOUT,
    )
    content = ""
    if resp.choices:
        content = resp.choices[0].message.content or ""
    parsed = extract_jso

In [18]:
# ============================================================
# Cell 18：对外接口 process_image —— 结构化诊断 + 字段兜底
# ============================================================
print("=" * 55)
print("  process_image —— 对外接口")
print("=" * 55)
print(inspect.getsource(mm.process_image))
print("重点：")
print("  - 先降采样再送 VLM（与入库口径一致：像素 + 字节双约束）")
print("  - OCR 失败不致命：回退让 VLM 自己看图识字")
print("  - error_type 白名单兜底：不在 网络/权限/软件/系统 里的，一律归为'系统'")
print("    → 保证下游拿到的分类永远是可枚举的，不会出现'乱码分类'")

  process_image —— 对外接口
async def process_image(image_bytes: bytes) -> dict:
    """图片处理：OCR + 视觉理解（结构化截图诊断）。

    返回字段：``ocr_text / description / error_code / service / hint / error_type``。
    """
    result: dict[str, Any] = {
        "ocr_text": "",
        "description": "",
        "error_code": "",
        "service": "",
        "hint": "",
        "error_type": "",
    }
    if not image_bytes:
        result["error_type"] = "系统"
        result["hint"] = "图片内容为空。"
        result["description"] = "未能读取图片内容。"
        return result

    # 1. 降采样（与入库口径一致：像素 + 字节双约束），并转为 data URL
    try:
        data, _ = await asyncio.to_thread(
            downsample,
            image_bytes,
            settings.EMBEDDING_MAX_IMAGE_SIDE,
            settings.EMBEDDING_MAX_IMAGE_BYTES,
        )
    except Exception as exc:
        logger.warning("图片降采样失败，使用原图：%s", exc)
        data = image_bytes
    data_url = build_data_url(data)

    # 2. OCR（local 时先本地 OCR；api 或本地不可用时由 VLM 完成）
    ocr_text = 

In [19]:
# ============================================================
# Cell 19：配图描述接口 describe_knowledge_image（入库期，失败不阻断）
# ============================================================
print("=" * 55)
print("  describe_knowledge_image —— 知识库配图描述（入库期）")
print("=" * 55)
print(inspect.getsource(mm.describe_knowledge_image))
print("重点：")
print("  - VLM 未配置 / 调用失败 → 返回空串，由调用方回退页正文，绝不 raise")
print("  - 描述会被写进【唯一文本向量空间】参与检索（第 36 节主线）")
print("  - 图片原图只落盘、不参与检索")

  describe_knowledge_image —— 知识库配图描述（入库期）
async def describe_knowledge_image(image_bytes: bytes) -> str:
    """为知识库配图生成一段客观描述（入库期调用，见《多模态 RAG 文本优先改造方案》）。

    描述用于把图片内容转成文本，写入唯一文本集合参与检索与作答（图片原图不参与检索）。
    VLM 未配置或调用失败时**返回空串**，由调用方回退页正文，绝不阻断入库。
    """
    if not image_bytes:
        return ""
    try:
        data, _ = await asyncio.to_thread(
            downsample,
            image_bytes,
            settings.EMBEDDING_MAX_IMAGE_SIDE,
            settings.EMBEDDING_MAX_IMAGE_BYTES,
        )
    except Exception as exc:  # noqa: BLE001
        logger.warning("配图描述降采样失败，使用原图：%s", exc)
        data = image_bytes
    try:
        return await asyncio.to_thread(_describe_sync, build_data_url(data))
    except PerceptionError as exc:
        logger.warning("配图描述跳过：%s", exc)
        return ""
    except Exception as exc:  # noqa: BLE001
        logger.warning("配图描述调用异常：%s", exc)
        return ""

重点：
  - VLM 未配置 / 调用失败 → 返回空串，由调用方回退页正文，绝不 raise
  - 描述会被写进【唯一文本向量空间】参与检索（第 36 节主线）
  - 图

### 5.4 VLM 幻觉与应对思路

多说一句"看不见的内容"是多模态模型的通病（幻觉）。本项目的应对是**双保险**：
1. **检索阶段提供事实依据**：作答基于知识库检索到的真实片段，而不是让模型凭空看图；
2. **生成阶段标注来源**：回复附"参考来源"，用户可回溯。

此外，配图描述提示词里明确写了"**不要臆测图中不存在的信息；不确定的项可省略**"，
这是从提示词层面压制幻觉的工程手段。

---
## Part 6：媒体文件全生命周期 —— 降采样 → 落盘 → 鉴权访问
---

### 6.1 为什么媒体不能像普通静态文件那样放出去？

用户上传的截图/语音可能包含隐私。如果做成静态目录（谁猜到 URL 谁就能下），就是数据泄露。
本项目采用的方案是**三段式**：

```
① 降采样 downsample()      ② 落盘 save_media()         ③ 鉴权访问
控制体积与成本        →     写进 media/，拿到 asset_id  →   /api/media/{id}（须登录）
```

### 6.2 第一段：降采样（双约束）

VLM / 嵌入接口对图片有**两个上限**：长边像素、编码后字节数。
`downsample()` 保证输出同时满足两者：
1. 长边压到 `EMBEDDING_MAX_IMAGE_SIDE`（1024）以内；
2. 优先输出 PNG（无损，适合带文字的截图）；超出 `EMBEDDING_MAX_IMAGE_BYTES`（32000 字节）时转 JPEG 并逐级降质量；
3. 仍超限就逐级缩小像素。

In [20]:
# ============================================================
# Cell 20：降采样源码 downsample()
# ============================================================
from core.rag.parsers import downsample

print("=" * 55)
print("  downsample —— 图片降采样（像素 + 字节双约束）")
print("=" * 55)
print(inspect.getsource(downsample))

  downsample —— 图片降采样（像素 + 字节双约束）
def downsample(data: bytes, max_side: int, max_bytes: int = 0) -> tuple[bytes, str]:
    """图片降采样：既压长边，也压字节数。

    嵌入 / 视觉理解接口对图片有**双重上限**：长边像素与编码后字节数
    （实测硅基流动图片输入上限 32KB）。本函数保证输出同时满足两者：

    1. 长边压到 ``max_side`` 以内；
    2. 优先输出 PNG（无损，适合含文字的截图）；超出 ``max_bytes`` 时改用 JPEG
       并逐级降质量；
    3. 仍超限则逐级缩小像素，直至满足上限。

    输入本身已满足两项约束且为 PNG/JPEG 时直接透传，避免重复有损编码。
    返回 ``(bytes, ext)``；非法图片抛 :class:`ParseError`。
    """
    try:
        from PIL import Image

        image = Image.open(io.BytesIO(data))
        image.load()
        src_format = (image.format or "").lower()
    except Exception as exc:
        raise ParseError(f"图片解码失败，无法降采样：{exc}") from exc

    try:
        oversize_pixels = bool(max_side) and max(image.size) > max_side
        oversize_bytes = bool(max_bytes) and len(data) > max_bytes
        if not oversize_pixels and not oversize_bytes and src_format in ("png", "jpeg"):
            # 已合规，直接透传，避免二次编码损失
            return data, "png" if s

In [21]:
# ============================================================
# Cell 21：拿真实素材实测降采样效果
# ============================================================
from pathlib import Path
from core.config import settings

candidates = [
    "docs/素材/images/vpn_error_691.png",
    "docs/素材/images/network_topology.png",
    "docs/素材/images/mail_config.png",
]
target = next((Path(p) for p in candidates if Path(p).exists()), None)

print("=" * 55)
print("  降采样实测")
print("=" * 55)
print(f"  配置：长边 <= {settings.EMBEDDING_MAX_IMAGE_SIDE}px，字节 <= {settings.EMBEDDING_MAX_IMAGE_BYTES}")

if target is None:
    print("  未找到素材图片，跳过实测。")
else:
    raw = target.read_bytes()
    out, ext = downsample(raw, settings.EMBEDDING_MAX_IMAGE_SIDE, settings.EMBEDDING_MAX_IMAGE_BYTES)
    print(f"  素材: {target.name}")
    print(f"  原始: {len(raw):>7} 字节")
    print(f"  降采样: {len(out):>7} 字节，输出格式 = {ext}")
    ratio = len(out) / len(raw) * 100
    print(f"  压缩到原来的 {ratio:.1f}%")

    if len(out) <= settings.EMBEDDING_MAX_IMAGE_BYTES and len(raw) > len(out):
        print("  [OK] 同时满足像素与字节上限")
    else:
        print("  [提示] 原图可能本来就在上限内（直接透传）")

  降采样实测
  配置：长边 <= 1024px，字节 <= 32000
  素材: vpn_error_691.png
  原始:   26187 字节
  降采样:   26187 字节，输出格式 = png
  压缩到原来的 100.0%
  [提示] 原图可能本来就在上限内（直接透传）


### 6.3 第二段：落盘与 asset_id 设计

In [22]:
# ============================================================
# Cell 22：落盘与寻址源码
# ============================================================
from core.infra.media import ext_from_mime, media_file, media_url, save_media

print("=" * 55); print("  save_media —— 落盘并返回 asset_id"); print("=" * 55)
print(inspect.getsource(save_media))
print("=" * 55); print("  media_file —— 按 asset_id 找回文件"); print("=" * 55)
print(inspect.getsource(media_file))
print("=" * 55); print("  media_url / ext_from_mime"); print("=" * 55)
print(inspect.getsource(media_url))
print(inspect.getsource(ext_from_mime))

  save_media —— 落盘并返回 asset_id
def save_media(data: bytes, ext: str, prefix: str = "asset") -> str:
    """保存二进制并返回 asset_id。

    参数：
        data: 文件字节内容。
        ext: 扩展名（可带或不带前导点，如 "png" 或 ".png"）。
        prefix: 文件名前缀，用于区分用途（如 image/audio，默认 asset）。
    返回：
        asset_id（形如 "asset-xxxxxxxxxxxx"，不含扩展名）。
    说明：
        只返回 asset_id 而不返回完整文件名，是为了让"对外标识"与"具体扩展名"解耦。
        磁盘上真实文件名为 "{asset_id}.{ext}"。
    """
    # 归一化扩展名：转小写并去掉可能的前导点，避免出现 "png." 之类的怪名字。
    ext = (ext or "bin").lower().lstrip(".")
    asset_id = f"{prefix}-{uuid.uuid4().hex[:12]}"
    directory = settings.media_dir()
    # 目录不存在就先建（mkdir -p），保证本地开发首次上传也能成功。
    directory.mkdir(parents=True, exist_ok=True)
    (directory / f"{asset_id}.{ext}").write_bytes(data)
    return asset_id

  media_file —— 按 asset_id 找回文件
def media_file(asset_id: str) -> Optional[Path]:
    """按 asset_id 查找媒体文件（扩展名不定，做前缀匹配）。

    参数：
        asset_id: 保存时返回的媒体标识。
    返回：
        找到则返回文件 Path；参数非法或找不到返回 None。
    说明：
        保存时扩展名不定，所以用 

**两个关键设计**（源码注释里也写了）：

1. **asset_id 与文件名解耦**：磁盘文件名是 `{asset_id}.{ext}`，但 `asset_id` 不含扩展名。
   原因：上传时浏览器只给 MIME，扩展名要落盘时才知道；查找时用 `asset_id.*` 前缀匹配即可。
2. **路径穿越防护**：`media_file` 在 `glob` 之前先检查 `asset_id` 是否含 `/`、`\`、`..`，
   直接拒绝。否则攻击者可以传 `../../etc/passwd` 跳出媒体目录读到系统文件。

### 6.4 实测：落盘 → 寻址 → 清理

In [23]:
# ============================================================
# Cell 23：媒体落盘 → 寻址 → 清理（真实调用）
# ============================================================
from core.infra.media import save_media, media_file, media_url
import os

if target is None:
    print("  无素材，跳过。")
else:
    asset_id = save_media(out, ext, prefix="image")
    print(f"  asset_id : {asset_id}")
    print(f"  对外 URL : {media_url(asset_id)}")
    found = media_file(asset_id)
    print(f"  寻回文件 : {found}")
    print(f"  文件大小 : {found.stat().st_size if found else 0} 字节")

    # 演示路径穿越防护：非法 asset_id 应被拒绝
    print(f"  非法 id 测试 ../secret → {media_file('../secret')}")

    # 清理本次演示产物
    if found and found.exists():
        os.remove(found)
        print("  已清理演示文件。")

  asset_id : image-6312ddd29cf7
  对外 URL : /api/media/image-6312ddd29cf7
  寻回文件 : data\media\image-6312ddd29cf7.png
  文件大小 : 26187 字节
  非法 id 测试 ../secret → None
  已清理演示文件。


### 6.5 第三段：鉴权访问（在路由层收口）

媒体文件**不挂静态目录**，而是必须经过 API 鉴权后再流式返回。

In [24]:
# ============================================================
# Cell 24：上传 / 读取两个端点的源码
# ============================================================
from core.api import chat_api

print("=" * 55)
print("  POST /api/media/upload —— 上传（限制体积、推断 modality）")
print("=" * 55)
print(inspect.getsource(chat_api.upload_media))
print("=" * 55)
print("  GET /api/media/{asset_id} —— 鉴权读取（未登录 401）")
print("=" * 55)
print(inspect.getsource(chat_api.get_media))
print("重点：")
print("  - Depends(require_role('user','agent','admin'))：进入函数前已校验登录+角色")
print("  - 前端 <img>/<audio> 无法带 Authorization 头，媒体 URL 通过 query 参数传 token")

  POST /api/media/upload —— 上传（限制体积、推断 modality）
@router.post("/media/upload")
async def upload_media(
    file: UploadFile = File(...),
    user: dict = Depends(require_role("user", "agent", "admin")),
) -> dict:
    """上传媒体文件（图片或语音）。

    调用方：各端发送带图/语音消息前先上传。
    入参：multipart 表单字段 file（UploadFile），File(...) 表示该字段必填。
    鉴权：允许 user / agent / admin。
    错误码：400 —— 文件为空或超过 settings.KB_MAX_FILE_MB 大小限制。
    返回：{"asset_id", "url", "modality"}；modality 按 MIME 类型推断为 image / voice。
    """
    # 依赖注入已在进入函数体前完成登录与角色校验，因此此处读文件时调用者必定合法
    data = await file.read()
    if not data:
        raise HTTPException(status_code=400, detail="上传文件为空")
    if len(data) > settings.KB_MAX_FILE_MB * 1024 * 1024:
        raise HTTPException(status_code=400, detail=f"文件超过 {settings.KB_MAX_FILE_MB}MB 限制")

    content_type = file.content_type or ""
    if content_type.startswith("image/"):
        modality, prefix = "image", "img"
    elif content_type.startswith("audio/"):
        modality, prefix = "voice", "

---
## Part 7：完整链路 + 随堂练习（含答案）
---

### 7.1 统一封装的价值（本节回扣）

```
用户上传语音 ──► /api/media/upload ──► save_media() ──► asset_id
                                          │
用户发消息(带 asset_id) ──► 业务层 ──► process_voice(audio_bytes) ──► text
用户上传截图 ──► 业务层 ──► process_image(image_bytes) ──► {error_code, error_type, ...}
AI 回复    ──► 业务层 ──► synthesize_speech(text) ──► (mp3_bytes, "mp3")
```

下游业务**永远只看到 3 个函数**，不关心输入来自语音还是图片、用的是哪家供应商。
新增一种感知能力，只需在工具层加一个函数——这就是"可维护"的基础。

### 7.2 随堂练习一：为 TTS 增加一个 provider 分支

`TTS_PROVIDER` 已经预留，但代码里还没用上。请你照 ASR 的写法，
给 `synthesize_speech` 设计一个"按 provider 选引擎"的骨架。

In [25]:
# ============================================================
# Cell 25：练习一参考思路（含答案）
# ============================================================
exercise_answer = """
async def synthesize_speech(text, voice=None):
    if not text or not text.strip():
        raise PerceptionError("语音合成失败：待合成文本为空。")
    chosen_voice = voice or settings.TTS_VOICE
    provider = (settings.TTS_PROVIDER or "edge").lower()

    if provider == "edge":
        audio = await asyncio.to_thread(_tts_sync, text, chosen_voice)
        return audio, "mp3"

    # 将来接第二家引擎：在这里加分支即可，返回 (bytes, ext)
    raise PerceptionError(f"未知的 TTS_PROVIDER：{provider}（当前仅支持 edge）")
"""
print(exercise_answer)
print("参考答案要点：")
print("  1. 只改 synthesize_speech 内部，接口签名 (text, voice) -> (bytes, ext) 不变")
print("  2. 新引擎返回自己的真实扩展名（可能是 wav），调用方无需改动")
print("  3. 未知 provider 抛可读错误，而不是静默走 edge")


async def synthesize_speech(text, voice=None):
    if not text or not text.strip():
        raise PerceptionError("语音合成失败：待合成文本为空。")
    chosen_voice = voice or settings.TTS_VOICE
    provider = (settings.TTS_PROVIDER or "edge").lower()

    if provider == "edge":
        audio = await asyncio.to_thread(_tts_sync, text, chosen_voice)
        return audio, "mp3"

    # 将来接第二家引擎：在这里加分支即可，返回 (bytes, ext)
    raise PerceptionError(f"未知的 TTS_PROVIDER：{provider}（当前仅支持 edge）")

参考答案要点：
  1. 只改 synthesize_speech 内部，接口签名 (text, voice) -> (bytes, ext) 不变
  2. 新引擎返回自己的真实扩展名（可能是 wav），调用方无需改动
  3. 未知 provider 抛可读错误，而不是静默走 edge


### 7.3 随堂练习二：新增一种感知能力

假设产品说："用户上传的 PDF 也要能直接问。" 你需要在感知工具层怎么扩展？

> 提示：先想清楚它属于哪条链路（是 ASR？OCR？还是**文档解析**？），再决定放哪个模块。

In [26]:
# ============================================================
# Cell 26：练习二参考思路（含答案）
# ============================================================
answer = """
结论：PDF 属于"文档解析"，不属于感知层（ASR/OCR/VLM/TTS）。

原因：感知层解决的是"非结构化信号 → 文本/结构化结论"（语音、截图、图像）；
而 PDF 是"结构化文档"，它的处理是 第 36 节的 core/rag/parsers.py 统一解析管线
（拆页、抽文本/图片/表格 → 多模态分块 → 嵌入入库），与 OCR/TTS 不是一回事。

所以扩展点应放在 parsers.py，而不是 multimodal.py。
把不同职责放进不同模块，才不会把感知层写成"万能工具箱"。
"""
print(answer)


结论：PDF 属于"文档解析"，不属于感知层（ASR/OCR/VLM/TTS）。

原因：感知层解决的是"非结构化信号 → 文本/结构化结论"（语音、截图、图像）；
而 PDF 是"结构化文档"，它的处理是 第 36 节的 core/rag/parsers.py 统一解析管线
（拆页、抽文本/图片/表格 → 多模态分块 → 嵌入入库），与 OCR/TTS 不是一回事。

所以扩展点应放在 parsers.py，而不是 multimodal.py。
把不同职责放进不同模块，才不会把感知层写成"万能工具箱"。



### 7.4 本节小结

| 链路 | 项目函数 | 核心设计 |
|------|---------|---------|
| ASR | `process_voice` | provider 抽象 + 失败抛异常（不返回错误文案） |
| TTS | `synthesize_speech` | 空文本早失败 + 扩展名由引擎返回 + 临时文件清理 |
| OCR | `_ocr_local_sync` | 只负责抠字，2.x/3.x 接口兼容 |
| VLM 诊断 | `process_image` | 结构化 JSON + `error_type` 白名单兜底 |
| VLM 配图 | `describe_knowledge_image` | 失败返回空串，绝不阻断入库 |
| 媒体 | `save_media` / `media_file` | asset_id 与扩展名解耦 + 路径穿越防护 |

### 7.5 交付物 / 验收

- 语音 → 文本可回显（`process_voice`）；
- 文本 → 语音 + 扩展名（`synthesize_speech`）；
- 截图 → 结构化诊断字段（`process_image`）；
- 媒体可经**带鉴权**的 URL 加载（`/api/media/{asset_id}`）；
- `tests/test_multimodal_asr.py` 全绿（4 例：空音频 / API 失败 / 空文本 / 正常去空白）。

### 7.6 下节课预告

**第 36 节：多模态与 RAG / 智能体深度融合开发**
- 统一解析管线 `parsers.py`：md/txt/pdf/docx/图片怎么变成"页与块"；
- 文本优先嵌入 `embedding.py`：为什么图片不直接嵌入；
- 单集合向量库 `vector_store.py` + 检索编排 `multimodal_rag.py`；
- 检索如何封装成 Agent 可调用的工具。

---

## 📚 参考资料

- 课件：`第35节-多模态核心能力开发与代码实现.pptx`
- 代码：`core/rag/multimodal.py` —— ASR / OCR / VLM / TTS / 会话前置识别
- 代码：`core/infra/media.py` —— 媒体落盘、寻址、鉴权
- 代码：`core/rag/parsers.py` —— `downsample()` 图片降采样
- 代码：`core/ai/llm.py` —— `extract_json()` 容错解析
- 测试：`tests/test_multimodal_asr.py`

> 🎉 恭喜你完成了本节课的学习！
> 现在你不仅知道多模态能力怎么调，更知道**为什么要把它们封装成一条工具层**。